In [ ]:
# Switch path to root of project
import os
os.environ["CUDA_VISIBLE_DEVICES"]="1"
current_dir = "/home/xxx/research/MedAlign"
# src_path = os.path.join(current_dir, 'src')
os.chdir(current_dir)

# from open_clip import create_model_and_transforms, get_mean_std
from open_clip import create_model_and_transforms, get_mean_std, HFTokenizer
from PIL import Image
import torch
from urllib.request import urlopen


model = 'ViT-B-16-quickgelu' # available pretrained weights ['ViT-L-14-336-quickgelu', 'ViT-B-16-quickgelu']
# pretrained = "/data//CLIP/unimed_clip_vit_l14_base_text_encoder.pt" # Path to pretrained weights
pretrained = "/data//CLIP/unimed_clip_vit_b16.pt"
text_encoder_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract" # available pretrained weights ["microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract", "microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract"]
mean, std = get_mean_std()
device='cuda'

In [ ]:
model_med, _, preprocess = create_model_and_transforms(
    model,
    pretrained,
    precision='amp',
    device=device,
    force_quick_gelu=True,
    pretrained_image=False,
    mean=mean, std=std,
    inmem=True,
    text_encoder_name=text_encoder_name,
)
tokenizer = HFTokenizer(
    text_encoder_name,
    context_length=256,
    **{},
)

In [4]:
model_visual_encoder = model_med.visual.cuda()
model_text_encoder = model_med.text_encoder.cuda()

In [ ]:
# load abdomen vqa data
import json
# vqa_train_file = "/data//hallucination/CARES/IU_Xray/training.json"
# vqa_train_file = "/data//hallucination/CARES/OmniMedVQA/training_masks_top4.json"
# vqa_train_file = "/data//hallucination/PathVQA/pvqa/training_masks_top4.json"
# vqa_train_file = "/data//hallucination/VQA_RAD/data/training_masks_top4.json"
# vqa_train_file = "/data//hallucination/Slake/data/training_masks_all.json"

vqa_train_file = "/data//hallucination/IU_Xray/data_report/training.json"
# vqa_train_file = "/data//hallucination/MIMIC_CXR/data_report/training.json"


with open(vqa_train_file, "r") as f:
    vqa_data = json.load(f)
# Check the loaded data
print(f"Loaded {len(vqa_data)} vqa data")
# filter only english data
vqa_data_train = vqa_data
print(f"Filtered {len(vqa_data_train)} vqa data")

Loaded 2069 vqa data
Filtered 2069 vqa data


In [ ]:
# sample images
import os
import random
from pathlib import Path

from sklearn.manifold import TSNE
import numpy as np

import matplotlib.pyplot as plt

# root_dir = Path("/data//hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data//hallucination/OmniMedVQA/VQA/raw/OmniMedVQA")
# root_dir = Path("/data//hallucination/PathVQA/pvqa/images/train")
# root_dir = Path("/data//hallucination/Slake/imgs")
# root_dir = Path("/data//hallucination/VQA_RAD/images/")
# image_paths = list(root_dir.rglob("source.jpg"))

root_dir = Path("/data//hallucination/IU_Xray/iu_xray/images")
# root_dir = Path("/data//hallucination/MIMIC_CXR/sampled_files_train")

image_paths = list(set([i['image'] for i in vqa_data_train]))

# Step 2: Randomly sample 100 images
sampled_paths = image_paths

print(f"Found {len(image_paths)} training images, sampled {len(sampled_paths)}")

Found 2069 training images, sampled 2069


In [7]:
full_sampled_paths = []
for path in sampled_paths:
    full_path = os.path.join(root_dir, path)
    full_sampled_paths.append(full_path)

In [8]:
import torch.nn.functional as F
from tqdm import tqdm

In [9]:
def interpolate_attention_map(attn_map: torch.Tensor, src_H: int = 14, tgt_H: int = 24) -> torch.Tensor:
    assert attn_map.shape[0] == src_H * src_H

    # Step 1: reshape to [1, 1, H, W]
    attn_2d = attn_map.view(1, 1, src_H, src_H)

    # Step 2: interpolate to target resolution
    attn_interp = F.interpolate(attn_2d, size=(tgt_H, tgt_H), mode='bilinear', align_corners=False)

    # Step 3: flatten to [tgt_H * tgt_H]
    return attn_interp.view(tgt_H * tgt_H)

In [10]:

attn_map_dict = {}
start_index, sample_number = 0, len(sampled_paths)
ind = 0
cos_matrix_dict = {}
for img_path in tqdm(full_sampled_paths[start_index:start_index+sample_number]):  # label: organ/lesion/other
    # image = Image.open(img_path).convert("RGB")
    # image_size = image.size[::-1]  # (H, W)
    # print(f"Processing {img_path}")
    relative_path = sampled_paths[ind]
    ind += 1
    inputs = preprocess(Image.open(img_path)).to("cuda").unsqueeze(0)

    # Preprocess and get patch tokens
    # inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        vision_output = model_visual_encoder.forward_intermediates(inputs)
        # print(vision_output['image_intermediates'][0].size())
        # patch_tokens = vision_output['image_intermediates'][-1][:, 1:, :].squeeze(0)  # [576, D]
        attention_map = vision_output['attn_weights']
    layer_idx = -1
    attn_map = attention_map[layer_idx].squeeze(0)  # [577, 577]

    # Extract [CLS] attention (row 0)
    cls_attn = attn_map[0]  # Shape [577]

    # Drop the [CLS] token itself (attention to itself)
    cls_to_patches = cls_attn[1:]  # Shape [576]

    # Normalize attention for visualization
    cls_to_patches = cls_to_patches / cls_to_patches.max()

    cls_to_patches = interpolate_attention_map(cls_to_patches, src_H=14, tgt_H=24)
    attn_map_dict[relative_path] = cls_to_patches.cpu().numpy()
    


100%|██████████| 2069/2069 [00:42<00:00, 49.15it/s]


In [16]:
# attn_map_dict.keys()

In [ ]:
# attn_map_dict['xmlab23/source.jpg'].shape

(576,)

In [ ]:
# save cosine similarity matrix
import pickle
# save_root_path = "/data//hallucination/CARES/IU_Xray/steering_data_base/"
# save_root_path = "/data//hallucination/CARES/OmniMedVQA/steering_data_base/"
# save_root_path = "/data//hallucination/PathVQA/pvqa/steering_data_base/"
# save_root_path = "/data//hallucination/Slake/steering/steer_data_base/"
# save_root_path = "/data//hallucination/VQA_RAD/steering/steer_data_base/"
save_root_path = "/data//hallucination/IU_Xray/data_report/steering_data_base/"
# save_root_path = "/data//hallucination/MIMIC_CXR/data_report/steering_data_base/"


if not os.path.exists(save_root_path):
    os.makedirs(save_root_path)
# save cosine similarity matrix
with open(save_root_path + "attn_map_dict.pkl", "wb") as f:
    pickle.dump(attn_map_dict, f)